In [ ]:
import requests
import json

url = 'http://localhost:8888'

def enqueue(org, repo, priority=50, depends_on=None):
    if depends_on is None:
        depends_on = []
    resp = requests.post(f'{url}/jobs', json={
        'org': org,
        'repo': repo,
        'priority': priority,
        'depends_on': depends_on,
    })
    print(resp.status_code, resp.text)
    return resp.json()

def get_status(job_id):
    resp = requests.get(f'{url}/jobs/{job_id}')
    print(resp.status_code, resp.text)
    return resp.json()

def get_trace(job_id):
    resp = requests.get(f'{url}/jobs/{job_id}/trace')
    print(resp.status_code, resp.text)
    return resp.text

def get_all_jobs():
    resp = requests.get(f'{url}/jobs')
    print(resp.status_code, resp.text)
    return resp.json()

In [ ]:
resp = get_all_jobs()
print(json.dumps(resp, indent=2))

200 {}
{}


In [ ]:
get_all_jobs()

200 {}


{}

In [ ]:
# resp = requests.post(f'{url}/jobs', json={
#     'org': "logger",
#     'repo': "logger",
#     'job_type': "git-clone",
# })
# print(resp.status_code, resp.text)

# resp = requests.post(f'{url}/jobs', json={
#     'org': "logger",
#     'repo': "logger",
#     'job_type': "clientlib-analysis",
# })
# print(resp.status_code, resp.text)


# repos = [
#     "logger"#, "bluwal", "secretmarket", "cell", "oilmarket", "crypter"
# ]
# for r in repos:
#     enqueue(r, r, priority=50)

200 {"job_id":"bd1159b8-53c8-4ade-b9b0-51e7d47fe40c","status":"queued"}


In [ ]:
# findings = [
#         {
#           "title": "Stack-based buffer overflow via unbounded scanf() in multiple input handlers",
#           "report": "## Overview\n\nThe binary `logger` reads user input using `scanf(\"%s\", ...)` in several places without specifying maximum field widths. This allows an attacker to provide an arbitrarily long string that will overflow the small stack buffers used to store IDs and secrets. Because this service is a network-facing binary (listening on TCP port 1781 via socat), an attacker can trigger these overflows remotely and potentially achieve control of the process (crash, memory corruption, or remote code execution) depending on the host protections in place.\n\n## Where it occurs\n\nThis occurs in the decompiled functions for the `logger` binary (file: `/opt/resources/logger`). Example vulnerable excerpts from the decompiled code:\n\n```c\n/* view_log */\nprintf(\"Enter log ID: \");\nfflush(NULL);\nscanf(\"%s\", local_1058); /* local_1058 is 32 bytes */\ngetchar();\nsnprintf(local_1038, 0x20, \"%s%s/log.txt\", \"./data/\", local_1058);\n\n/* view_unredacted */\nprintf(\"Enter log ID: \");\nfflush(NULL);\nscanf(\"%s\", local_20a8); /* local_20a8 is 32 bytes */\ngetchar();\nprintf(\"Enter redacted secret: \");\nfflush(NULL);\nscanf(\"%s\", local_2088); /* local_2088 is 32 bytes */\ngetchar();\n```\n\nThese snippets are found in the decompiled output and correspond to the interaction menu options used by remote clients.\n\n## Vulnerability Details\n\nThe format specifier \"%s\" for scanf reads input until whitespace but does not enforce any maximum length. The destination buffers (e.g. `local_1058`, `local_20a8`, `local_2088`) are small (32 bytes), so any input longer than the buffer length will overflow into adjacent stack memory. This can overwrite saved frame pointers, return addresses, or other local variables.\n\nAlthough the binary is compiled as a PIE and uses some modern protections (e.g. NX), the presence of an unbounded stack write remains a high-risk issue: with enough control over the overflow an attacker can corrupt the stack and pivot execution (for example by overwriting return addresses or function pointers). The binary also contains many useful libc calls (fopen, fgets, printf, system-like primitives) that make constructing a reliable exploit feasible for a motivated attacker.\n\n## Impact\n\n- Remote arbitrary code execution (RCE) or control-flow hijacking on the service process.\n- Remote compromise of the host or container where `logger` runs, potentially exposing all stored `./data/` logs and secrets.\n- Denial-of-service (process crash) by triggering an out-of-bounds write and crash.\n- Possible privilege escalation if the service runs with elevated privileges.\n\nBecause the service is network-exposed via socat (TCP port 1781), the vulnerability is remotely exploitable by any attacker that can reach the service.\n\n## Steps to Reproduce / Exploit\n\nHigh-level reproduction steps (do not execute against unauthorized targets):\n\n1. Connect to the TCP service (host:port) the binary is listening on, e.g. with netcat/socat: `nc <host> 1781`.\n2. When prompted by the menu, choose the option that triggers the vulnerable scanf (e.g. option 2 \"View a previous log\" or option 3 \"View an unredacted log\").\n3. Enter an oversized string (several hundred to thousands of characters) when asked for the Log ID or the redacted secret. Example (conceptual):\n\n```\n2\\nA...A\\n\n# where A...A is 2000 'A' bytes\n```\n\n4. Observe crash or abnormal behavior. A successful exploit would craft the overflow to overwrite saved return addresses and chain a ROP payload (considering PIE, NX, RELRO mitigations) or otherwise hijack execution.\n\nNotes about exploitability: the binary is PIE and linked with musl; it uses full relocations (BIND_NOW), NX, and RELRO which raise difficulty of exploitation but do not eliminate the risk. A skilled attacker can still mount ROP-based attacks or leverage information leaks to defeat ASLR.\n\n## Remediation\n\n- Replace all uses of `scanf(\"%s\", ...)` with safe input APIs that limit the number of characters read (for example `fgets(buf, sizeof buf, stdin)` and then strip the trailing newline). Or use a width-limited scanf: `scanf(\"%31s\", buf)` when the buffer has 32 bytes.\n- Validate and enforce maximum lengths for all user-controlled strings before using them in sensitive APIs.\n- Prefer memory-safe parsing libraries or languages for network-facing parsers.\n- Consider hardening compile-time protections (stack canaries, PIE, RELRO, least privilege) and runtime mitigations (dropping privileges, running in a sandbox) to reduce impact if other bugs exist.\n\n```c\n/* Suggested fix example */\nchar id[32];\nif (fgets(id, sizeof(id), stdin) != NULL) {\n    id[strcspn(id, \"\\n\")] = '\\0';\n}\n```\n\nThis issue is straightforward and high-risk; treat it as critical and patch immediately.\n",
#           "summary": "Multiple uses of scanf(\"%s\", ...) read unbounded input into small fixed-size buffers (32 bytes), allowing remote stack-based buffer overflows and potential RCE via the network-facing service.",
#           "file_path": "logger",
#           "confidence": "high",
#           "severity": "critical",
#           "exploitability": "high"
#         },
#         {
#           "title": "Predictable ID/secret generation via insecure srand()/rand() seeding",
#           "report": "## Overview\n\nThe binary generates both the public log ID (directory name) and the per-log secret using the C standard library PRNG (srand()/rand()) and seeds it with predictable values (current time, process id and an internal counter). This produces non-cryptographically-random identifiers and secrets that can be predicted or brute-forced by an attacker. Secrets used to authorize unredaction are therefore not securely generated.\n\n## Where it occurs\n\nFunction `generate_id` in the `logger` binary (decompiled) seeds the PRNG and formats the output into a string buffer:\n\n```c\nvoid generate_id(char *param_1,size_t param_2)\n{\n  time_t tVar3 = time(NULL);\n  uint uVar1 = getpid();\n  uVar1 = counter ^ (uint)tVar3 ^ uVar1;\n  counter = counter + 1;\n  srand(uVar1);\n  int iVar2 = rand();\n  snprintf(param_1, param_2, \"%012ld\", (long)iVar2 % 1000000000000);\n}\n```\n\nThis function is called twice in `save_log()` to create a public directory ID and the corresponding secret value written to the log file first line and shown to the user.\n\n## Vulnerability Details\n\n- The implementation uses the non-cryptographic PRNG `rand()` seeded via `srand()`.\n- The seed is computed as a simple XOR of `time(NULL)`, `getpid()`, and an in-memory `counter` value. These values are either predictable (current time) or have a small entropy space (PID, counter).\n- Using this predictable seeding, an attacker who knows or can guess the approximate time a log was created (or can observe a related event) can re-seed an offline PRNG to reproduce the same sequence of rand() outputs and thus determine both the public ID and the secret.\n\nBecause the secret is used as the authentication token for viewing the unredacted log, a predicted secret allows bypassing the authentication check and reconstructing sensitive log contents.\n\n## Impact\n\n- An attacker that can guess or determine the approximate generation time (which is often trivial for temporally-correlated actions) can predict the secret and hence perform `view_unredacted` to recover the original unredacted log content.\n- If an attacker can predict or enumerate directory IDs, they may be able to access or enumerate stored logs.\n- Secrets created with this weak randomness are also vulnerable to offline brute-force where the entropy is low (rand() typically returns 31 bits or less).\n\nThis substantially weakens the intended secrecy of logs and allows unauthorized access to sensitive stored data.\n\n## Steps to Reproduce / Exploit\n\n1. Observe or estimate the time (wall-clock time) when logs are created. This could be done by creating your own logs and comparing times or by making repeated requests until the service responds.\n2. Reconstruct the seed computation in a local program using candidate times, known PIDs (musl often sets predictable PIDs in containers), and counter guesses (0,1,...).\n3. Seed a local PRNG using srand(seed) and call rand() to compute the expected outputs; format them with the same snprintf rules.\n4. If the predicted secret matches the secret stored in a target's log's first line, request `view_unredacted` supplying the guessed secret and the public ID to retrieve full unredacted contents.\n\nA simple brute-force script can iterate over a small time window and counter values to recover secrets quickly.\n\n## Remediation\n\n- Do not use `srand()`/`rand()` for generating secrets, IDs, or tokens. Use a cryptographically secure random source such as /dev/urandom, getrandom(), or platform CSPRNG APIs (e.g., arc4random, getrandom, OpenSSL RAND_bytes).\n\nExample (POSIX) using getrandom()/getentropy():\n\n```c\nunsigned char buf[16];\nif (getrandom(buf, sizeof(buf), 0) == sizeof(buf)) {\n    // convert to hex or base64 for ID/secret\n}\n```\n\n- Increase entropy (longer secrets) and do not expose raw secrets unnecessarily.\n- If secrets must be human-readable, use a proper token generation library (cryptographic GUIDs or HMAC-based tokens) and enforce sufficient length (128-bit+).\n- Log the use of random tokens only in secure locations and avoid printing them unnecessarily.\n\nFixing the RNG usage will prevent offline prediction and brute-force of authentication tokens and protect log confidentiality.\n",
#           "summary": "generate_id() uses srand()/rand() seeded with time, PID, and a simple counter; this produces predictable IDs and secrets that can be guessed or brute-forced.",
#           "file_path": "logger",
#           "confidence": "high",
#           "severity": "medium",
#           "exploitability": "high"
#         },
#         {
#           "title": "Unsafe temporary file handling and rename (TOCTOU / symlink replacement)",
#           "report": "## Overview\n\nThe `logger` binary performs redaction by writing a redacted copy to a temporary filename and then renaming that temporary over the original file. The temporary filename is derived directly from the original path (``%s.tmp``). Combined with the service creating directories with permissive mode (0777), this pattern introduces race conditions (TOCTOU) and symlink-based replacement attacks. An attacker with write access to the `./data` tree (or the ability to control a path component via traversal) can influence what file is replaced, potentially causing arbitrary file overwrite or exposure of sensitive data.\n\n## Where it occurs\n\nDecompiled excerpt from `redact_log` (file: `/opt/resources/logger`):\n\n```c\nsnprintf(local_1058, 0x40, \"%s.tmp\", param_1);\n__stream_01 = fopen(local_1058, \"w\");\n... /* write redacted content to __stream_01 */\n...\nrename(local_1058, param_1);\n```\n\nAnd in `save_log` the directory is created with permissive mode:\n\n```c\nsnprintf(local_1158, 0x20, \"%s%s\", \"./data/\", local_1198);\nmkdir(local_1158, 0x1ff); /* 0x1ff == 0777 octal */\n```\n\n## Vulnerability Details\n\n- The temporary name is predictable (`<original_filename>.tmp`) and is created in the same directory as the target file.\n- The service performs `fopen` on that predictable temporary name without verifying that it is not a symbolic link or pointing to a sensitive path.\n- The service later calls `rename(tmpname, targetname)` replacing the target atomically.\n\nAn attacker that can control directory contents (or create entries under `./data`) can exploit a race between the service creating the temporary file and renaming it, for example by pre-creating a symlink or by swapping filesystem entries. This can cause the service to overwrite an arbitrary file that the running user has write permission to, or to disclose sensitive data by writing content to a file under the attacker’s control.\n\nThe use of `mkdir(..., 0x1ff)` (0777) creates directories that are writable by all users, which increases the risk if multiple users/processes share the same host or container filesystem.\n\n## Impact\n\n- Arbitrary file overwrite: an attacker could cause the service to write redacted content into an attacker-controlled location, overwriting files the service user can write.\n- Data exposure: if a symlink is used to point the temporary file to an attacker-accessible path, the attacker can obtain redacted or unredacted content.\n- Potential for privilege escalation or corruption of service state if critical files are overwritten.\n\nThe attackability depends on whether the attacker has the ability to create files or symlinks in the `./data` hierarchy (which is feasible if the directory permissions are overly permissive or a malicious user has local filesystem access). When the `./data` tree is mounted from the host (volume mount), a remote attacker or host user could exploit this.\n\n## Steps to Reproduce / Exploit\n\n1. Identify the target data directory; attempt to create a symlink named `./data/<id>/log.txt.tmp` pointing to an attacker-controlled file or a sensitive location that the service user can write to.\n2. Trigger `save_log` / `redact_log` for the corresponding log (or cause the service to create the temporary file). If the attacker can race the creation, swap the symlink in time so that the service opens and writes to the attacker-controlled path.\n3. After `rename(tmpname, targetname)` executes, the target file will point to or have been replaced by the attacker's chosen file.\n\nNote: Successful exploitation requires the attacker to have write ability in the `./data` path (or the service to accept user-controllable IDs that permit traversal into attacker-controlled directories).\n\n## Remediation\n\n- Use secure temporary file creation APIs (e.g., `mkstemp`) that create an exclusively-owned temporary file safely and return a file descriptor that cannot be hijacked via symlinks.\n\nExample:\n\n```c\nchar tmp_template[PATH_MAX];\nsnprintf(tmp_template, sizeof(tmp_template), \"%s.tmp.XXXXXX\", param_1);\nint fd = mkstemp(tmp_template);\nif (fd >= 0) {\n    FILE *f = fdopen(fd, \"w\");\n    // write to f, then close\n    rename(tmp_template, param_1);\n}\n```\n\n- Avoid predictable temporary filenames in the same location as the target.\n- Restrict directory permissions (do not use 0777); use a restrictive umask and ensure only the service account can write to its data directory.\n- Validate or sanitize user-controlled path components and do not allow arbitrary traversal outside intended directories.\n\nAddressing these issues will close a class of filesystem race and symlink attacks that can lead to data corruption or disclosure.\n",
#           "summary": "Predictable temporary filename usage (<original>.tmp) and world-writable directories (mkdir mode 0777) create TOCTOU/symlink risks allowing file overwrite or data exposure.",
#           "file_path": "logger",
#           "confidence": "high",
#           "severity": "medium",
#           "exploitability": "medium"
#         },
#         {
#           "title": "User-supplied regex in redaction leads to potential ReDoS / resource exhaustion",
#           "report": "## Overview\n\nThe `logger` binary accepts a user-supplied regular expression (regex) to redact log contents. The regex is compiled with `regcomp()` and then applied to each line using `regexec()` while iterating through entire log files. Unbounded or pathological regular expressions (or long input lines) can cause excessive CPU consumption (catastrophic backtracking) and allow an attacker to cause a denial of service (ReDoS) against the service.\n\n## Where it occurs\n\nRelevant decompiled excerpt from `/opt/resources/logger` in `redact_log`:\n\n```c\nregcomp(&local_1098, param_3, 1);  /* compile user-provided regex */\nfprintf(__stream_00, \"%s\\n\", param_3);\nwhile ((pcVar2 = fgets(local_1018, 0x1000, __stream)) != NULL) {\n    while (regexec(&local_1098, local_1018, 1, &local_10a8, 0) == 0) {\n        /* write match to secret file, replace matched bytes with '*' */\n        fprintf(__stream_00, \"%.*s\\n\", ...);\n        memset(..., '*', ...);\n    }\n    fputs(local_1018, __stream_01); /* write redacted line */\n}\nregfree(&local_1098);\n```\n\nThe regex pattern is read from user input without validation.\n\n## Vulnerability Details\n\n- POSIX regular expressions can be made pathological such that `regexec()` (or the regex engine in the C library) will spend exponentially increasing time processing certain input strings (catastrophic backtracking), especially for constructs like nested or ambiguous repetition.\n- The code applies the compiled regex to every line in the log and keeps applying it in a loop to find subsequent matches. If a single `regexec()` call is expensive, repeated application against large or crafted log lines will amplify CPU usage.\n- There is no timeout, complexity limit, or upper bound on the amount of time spent processing a single regex against a single line.\n\n## Impact\n\n- An attacker who can supply the regex pattern (via the save_log flow) can cause the service to spend excessive CPU time on redaction, potentially rendering the service unresponsive or consuming host resources.\n- This can be used as a denial-of-service (DoS) vector against the service, affecting availability for legitimate users.\n\n## Steps to Reproduce / Exploit\n\n1. Connect to the service and create a log (save_log). When prompted for the redaction regex, supply a pathological pattern such as one that triggers backtracking (examples depend on the specific regex engine; eg. patterns that use nested repetitions with ambiguous matches).\n2. Ensure there is a log line that triggers heavy backtracking (e.g. a very long line of repeated characters).\n3. Observe the service taking a long time or consuming CPU while handling redaction.\n\nNote: Exact pathological patterns depend on the C library's regex engine (musl in this binary). Regardless, complex repeating constructs can be problematic.\n\n## Remediation\n\n- Do not accept arbitrary regexes for server-side operations without safeguards. Options:\n  - Restrict allowed regex syntax (disallow nested quantifiers, backreferences, or other constructs known to cause exponential behavior).\n  - Limit maximum regex compile time and execution time (run `regexec()` in a worker thread and enforce a timeout; kill or skip if it exceeds a threshold).\n  - Limit the size of input lines to be scanned (shorter lines reduce impact).\n  - Use safer, bounded pattern matching (e.g., fixed substring searches or controlled search libraries) where possible.\n- Validate and sanitize the user-supplied regex before running it on large inputs.\n\nImplementing these mitigations reduces the risk of CPU exhaustion from attacker-supplied or malicious regular expressions and improves service availability and robustness.\n",
#           "summary": "The application compiles and applies user-supplied POSIX regex patterns to logs without any complexity checks or timeouts, enabling ReDoS (catastrophic backtracking) and CPU exhaustion.",
#           "file_path": "logger",
#           "confidence": "medium",
#           "severity": "medium",
#           "exploitability": "medium"
#         },
#         {
#           "title": "Sensitive data exposure: secrets and redaction content stored in world-writable files under ./data",
#           "report": "## Overview\n\nThe service stores sensitive values (the per-log secret and the redacted match data) on disk in the `./data/<id>/` directory. Directories are created with permissive permissions (0777) and the secret is written to files in plain text (`log.txt` first line, and `secret.txt` containing redacted matches). This design exposes sensitive information to any user or process that can access the filesystem (including host users when `./data` is volume-mounted), leading to disclosure of confidential log contents.\n\n## Where it occurs\n\n- `save_log()` writes the secret into the newly created log file:\n\n```c\nsnprintf(local_1158, 0x20, \"%s%s/log.txt\", \"./data/\", local_1198);\n__stream = fopen(local_1158, \"w\");\nfprintf(__stream, \"%s\\n\", local_1178); /* secret written as first line */\n...\n```\n\n- `redact_log()` writes the regex and matched substrings to the secret file:\n\n```c\n__stream_00 = fopen(param_2, \"w\");\nregcomp(&local_1098, param_3, 1);\nfprintf(__stream_00, \"%s\\n\", param_3); /* regex saved */\n/* for each match: */\nfprintf(__stream_00, \"%.*s\\n\", ...); /* matched substring saved */\n```\n\n- `save_log()` creates directories with permissive permissions:\n\n```c\nmkdir(local_1158, 0x1ff); /* 0x1ff == 0777 octal */\n```\n\n## Vulnerability Details\n\n- Secrets are stored unencrypted in filesystem files. Anyone with filesystem access can read them.\n- Directory permissions `0777` allow other local users or processes to add or modify files under the same `./data` tree, increasing the risk of data tampering, symlink attacks, or unauthorized reads.\n- When the service is run inside a container and `./data` is mapped to a host directory, host users can read the secret files directly.\n\n## Impact\n\n- Confidential log contents and redaction-specific secret tokens can be trivially read by any local user or by host users when the data directory is mounted, breaking confidentiality guarantees.\n- Disclosure of secrets enables unauthorized calls to `view_unredacted` (if the attacker can subsequently interact with the service) and direct reading of secret.txt messages.\n- The wide-open directory permissions also facilitate attacks that exploit the predictable temporary filename behavior (see related TOCTOU finding).\n\n## Steps to Reproduce / Exploit\n\n1. If you have filesystem access to the container or host filesystem where `./data` resides, list `./data` and inspect any `./data/<id>/log.txt` and `./data/<id>/secret.txt` files.\n2. Read the first line of `log.txt` to obtain the per-log secret. Read `secret.txt` to see the stored regex and matched substrings. Both are in plaintext.\n\nIf the `./data` directory is mounted as a Docker volume, run on host:\n\n```\ncat /path/to/volume/<id>/log.txt\ncat /path/to/volume/<id>/secret.txt\n```\n\n## Remediation\n\n- Do not store secrets or sensitive plaintext data in world-readable files. Use appropriate file permissions (e.g., `0700` for directories and `0600` for files) and ensure the service runs under a dedicated unprivileged user.\n- Encrypt sensitive data at rest if it must be persisted, or avoid persisting secrets where possible.\n- Limit the directory creation mode to a minimal set (e.g., `mkdir(path, 0700)`), and set an appropriate umask.\n- Consider storing secrets in a secure secrets store (Vault, OS-provided keyring) rather than the filesystem.\n\nApplying these changes reduces the chance of local or host-level attackers reading sensitive log secrets and retained redaction data.\n",
#           "summary": "The application writes per-log secrets and matched (redacted) substrings to files under ./data and creates directories as 0777; these plaintext files are readily readable by other local users or host processes when the volume is mounted.",
#           "file_path": "logger",
#           "confidence": "high",
#           "severity": "medium",
#           "exploitability": "high"
#         }
#       ] + [
#         {
#           "title": "Stack-based buffer overflow via unsafe scanf in view_log() and view_unredacted()",
#           "report": "## Overview\n\nThe native service `logger` reads user-provided strings using unbounded scanf(\"%s\", ...) into fixed-size stack buffers in multiple functions (notably view_log and view_unredacted). This allows an attacker to overflow those buffers, overwrite stack data, and cause crashes. In some builds/conditions this could be escalated to arbitrary code execution or information disclosure. The service is exposed over TCP (socat listens on port 1781) so this vulnerability is remotely reachable.\n\n## Where it occurs\n\n- Binary: `/opt/resources/logger` (decompiled output shown below)\n- Functions: `view_log()` and `view_unredacted()` in the decompiled source\n\nRelevant decompiled snippets:\n\n```c\n/* view_log */\nundefined1 local_1058[32];\nchar local_1038[32];\n...\nprintf(\"Enter log ID: \");\nfflush((FILE *)0x0);\nscanf(\"%s\", local_1058);\ngetchar();\nsnprintf(local_1038,0x20, \"%s%s/log.txt\", \"./data/\", local_1058);\n__stream = fopen(local_1038, \"r\");\n```\n\n```c\n/* view_unredacted */\nundefined1 local_20a8[32];\nchar local_2088[32];\n...\nprintf(\"Enter log ID: \");\nfflush((FILE *)0x0);\nscanf(\"%s\", local_20a8);\ngetchar();\nprintf(\"Enter redacted secret: \");\nfflush((FILE *)0x0);\nscanf(\"%s\", local_2088);\ngetchar();\n```\n\nBoth `local_1058` and `local_20a8`/`local_2088` are only 32 bytes long but `scanf(\"%s\", ...)` does not enforce any maximum length, allowing arbitrarily long input to be written to the stack.\n\n## Vulnerability Details\n\n- The calls to `scanf(\"%s\", buffer)` accept a whitespace-delimited token of any length and will write past the end of the provided stack buffer if the token is larger than the buffer.\n- The binary implements stack canary checks (presence of `__stack_chk_fail` in decompiled output), so simple overwrites may trigger an immediate crash. However, depending on the exact runtime environment (compiler flags, ASLR, PIE, NX), it may be possible to: crash the service (DoS), leak memory, or craft a more complex exploit (stack pivot/ROP or heap corruption) for code execution.\n- The service is network-facing (socat TCP-LISTEN:1781,fork,reuseaddr EXEC:\"/service/logger\"), so this flaw is remotely reachable by any network client that can connect to the TCP port.\n\n## Impact\n\n- Remote Denial of Service: a remote attacker can reliably crash the process by sending overly long inputs for the prompts that call `scanf(\"%s\",...)`.\n- Potential Remote Code Execution: with an appropriate environment (e.g., disabled/absent stack protections, or advanced memory corruption primitives), an attacker could escalate to arbitrary code execution.\n- Information Disclosure: even if code execution is not achieved, memory corruption could lead to unexpected reads/behavior leaking sensitive data.\n\n## Steps to Reproduce / Exploit\n\n1. Start the service locally (as the container does) or connect to the running service on port 1781 (e.g. `nc <host> 1781`).\n2. At the menu prompt, choose option `2` (View a previous log) or `3` (View an unredacted log).\n3. When prompted for `Enter log ID:` or `Enter redacted secret:`, send an input longer than 32 bytes (e.g., 1000 'A' characters). Example (netcat):\n\n```bash\n# Example using netcat (replace host:port appropriately)\necho -e \"2\\n$(python3 -c 'print(\"A\"*1000)')\\n\" | nc 127.0.0.1 1781\n```\n\n4. The service will likely crash or exhibit abnormal behavior. With further binary-specific exploit development (bypass stack canary, return-to-libc, ROP), a remote exploit could be constructed.\n\nNote: Because a stack protector is present, exploitation for code execution will require more advanced techniques, but the initial overflow and DoS are trivial to trigger.\n\n## Remediation\n\n- Replace unsafe scanf(\"%s\", ...) calls with bounded input functions. For example use `fgets(buffer, sizeof(buffer), stdin)` and strip the trailing newline, or use `scanf(\"%31s\", buffer)` to limit input length to buffer size minus 1 for the NUL byte.\n\nExample fix:\n\n```c\nchar id[32];\nif (fgets(id, sizeof(id), stdin) != NULL) {\n    id[strcspn(id, \"\\n\")] = '\\0';\n}\n```\n\n- Validate and canonicalize any path components constructed from user input before using them with filesystem APIs.\n- Run the service with appropriate exploit mitigations enabled (ASLR, stack canaries, NX) and consider running unprivileged within the container.\n",
#           "summary": "Unbounded scanf(\"%s\") calls in view_log() and view_unredacted() write arbitrary-length input into 32-byte stack buffers, allowing remote buffer overflow leading to DoS and possible RCE.",
#           "file_path": "/opt/resources/logger",
#           "confidence": "high",
#           "severity": "high",
#           "exploitability": "high"
#         },
#         {
#           "title": "Insecure redaction: regex-based redaction writes unredacted substrings to disk and enables ReDoS",
#           "report": "## Overview\n\nThe application implements a regex-based redaction routine that (1) compiles a user-supplied regex and uses it to locate substrings to redact, (2) writes the matched substrings into a separate file (`secret.txt`), and (3) replaces the matched bytes in the original log with asterisks. Two issues arise:\n\n- The redaction algorithm writes the sensitive substrings (the exact matches) to disk in plaintext (secret.txt). Those files live in the `./data` directory which is mounted out of the container by docker-compose, making the unredacted data persistently available on the host.\n- The server compiles user-provided regexes with no safeguards and applies them to arbitrary input, enabling expensive pathological patterns that can trigger catastrophic backtracking and denial-of-service (ReDoS).\n\n## Where it occurs\n\n- Binary: `/opt/resources/logger` (decompiled source)\n- Function: `redact_log()` (called from `save_log()`)\n\nDecompiled excerpt (relevant parts):\n\n```c\nvoid redact_log(char *param_1, char *param_2, char *param_3)\n{\n    ...\n    __stream_00 = fopen(param_2, \"w\"); // param_2 -> ./data/<id>/secret.txt\n    ...\n    regcomp(&local_1098, param_3, 1);\n    fprintf(__stream_00, \"%s\\n\", param_3); // write the regex itself to secret.txt\n    ...\n    while (fgets(local_1018, 0x1000, __stream)) {\n        while (regexec(&local_1098, local_1018, 1, &local_10a8, 0) == 0) {\n            fprintf(__stream_00, \"%.*s\\n\", match_len, local_1018 + match_start); // write matched substring\n            memset(local_1018 + match_start, 0x2a, match_len); // replace with '*' in output\n        }\n        fputs(local_1018, __stream_01);\n    }\n}\n```\n\n## Vulnerability Details\n\n1. Sensitive data exposure on disk: The implementation intentionally writes the exact substrings matched by the redaction regex into `secret.txt`. That file contains the regex on the first line and then one line per redaction match (the original unredacted text). These files are created under `./data/<id>/secret.txt` and are persisted, by default, via the docker-compose volume mapping (`./data:/service/data`). An attacker who can read files on the host or otherwise access the `data` directory can obtain all previously redacted content in plaintext.\n\n2. ReDoS / CPU exhaustion: `regcomp`/`regexec` are invoked on user-provided patterns without limits. Carefully crafted patterns (for example those with nested quantified subexpressions like `(a+)+` or heavy backtracking constructs) applied against log lines can cause excessive CPU usage and stall the process while redaction completes. Since redaction is performed synchronously during `save_log()`, this can be triggered remotely by a client saving a log with a malicious pattern.\n\n3. Logical issue: the program stores the *secret* used for access control in the first line of `log.txt` while storing the regex and matched substrings in `secret.txt`. The split design is confusing and increases the attack surface because the persistence of secret.txt contains sensitive data that may be overlooked when auditing backups or mounted volumes.\n\n## Impact\n\n- Data leakage: secrets and other sensitive substrings that should be redacted are kept in plaintext in `secret.txt` and therefore can be read if the attacker or other parties have filesystem access to the volume. Because docker-compose mounts `./data` on the host, these files are accessible on the host filesystem.\n\n- Denial of Service: a malicious user can submit a carefully crafted regex to tie up the process for an extended period, preventing normal operation for other clients.\n\n- Persistence of sensitive material: redacted text is still retained elsewhere on disk, undermining the very purpose of redaction.\n\n## Steps to Reproduce / Exploit\n\n1. Start the service (or connect to the running service on TCP port 1781).\n2. Select option `1. Save a new log`.\n3. Enter a log entry containing data you want to have redacted (for example a secret string `PASSWORD=supersecret`).\n4. When prompted `Enter regex pattern for redaction:`, provide a pattern that matches the sensitive content (e.g. `PASSWORD=[^\\n]+`). This will cause the matched substring `PASSWORD=supersecret` to be written into `./data/<id>/secret.txt`.\n\nTo demonstrate ReDoS:\n\n- Use a long input line such as `A` repeated several thousand times and submit a pathological regex like `(A+)+$` or `((A|A)+)+` as the redaction pattern. The `regexec`/`regcomp` calls will spend a long time processing the line, causing CPU exhaustion.\n\nBecause `./data` is a host-mounted volume (per docker-compose.yaml), the `secret.txt` file is visible on the host after the operation and can be read directly, exposing the matched substrings.\n\n## Remediation\n\n- Do not store original matched substrings in plaintext. If you must persist redaction metadata, store only non-sensitive metadata (e.g., positions or non-reversible hashes) or store the redaction pattern only — never the original sensitive text.\n\n- Limit or sanitize user-supplied regexes: impose a maximum length, forbid expensive constructs, or use a safe regex engine with execution timeouts or complexity limits.\n\n- Consider performing redaction in memory and never write unredacted fragments to persistent storage. If you must persist secrets, encrypt them with a server-side key that is not exposed through the public interface or file shares.\n\n- Use least-privilege filesystem permissions for `./data`: do not create files/directories world-writable or on host volumes without access controls.\n",
#           "summary": "The redaction routine writes matched (unredacted) substrings to ./data/<id>/secret.txt and accepts arbitrary user regexes, enabling persistent leakage of redacted data and ReDoS via expensive regex patterns.",
#           "file_path": "/opt/resources/logger",
#           "confidence": "high",
#           "severity": "medium",
#           "exploitability": "medium"
#         },
#         {
#           "title": "Path traversal / insufficient validation when constructing file paths from user-supplied log IDs",
#           "report": "## Overview\n\nThe service constructs file paths using user-supplied log IDs without validation or canonicalization. It concatenates the string into `./data/<ID>/log.txt` and `./data/<ID>/secret.txt` using `snprintf` but does not sanitize or restrict characters in `<ID>`. This allows path traversal (use of `..` and slashes) and other crafted IDs that may read or write files outside the intended data directory or otherwise manipulate file system locations.\n\n## Where it occurs\n\n- Binary: `/opt/resources/logger` (decompiled source)\n- Locations: `view_log()`, `view_unredacted()`, `save_log()` (path construction)\n\nDecompiled examples:\n\n```c\n/* view_log */\nprintf(\"Enter log ID: \");\nfflush((FILE *)0x0);\nscanf(\"%s\", local_1058);\ngetchar();\nsnprintf(local_1038,0x20, \"%s%s/log.txt\", \"./data/\", local_1058);\n__stream = fopen(local_1038, \"r\");\n\n/* view_unredacted */\nsnprintf(local_2068,0x20, \"%s%s/log.txt\", \"./data/\", local_20a8);\nsnprintf(local_2048,0x20, \"%s%s/secret.txt\", \"./data/\", local_20a8);\n```\n\nNo checks are performed to ensure `local_1058` / `local_20a8` are simple identifiers without path separators or traversal sequences.\n\n## Vulnerability Details\n\n- The code accepts arbitrary bytes for the ID (via `scanf(\"%s\")`) and directly appends it to `./data/`. An attacker can include `..` path components and `/` separators to traverse out of the data directory and reference file locations elsewhere in the filesystem (for example pointing at a directory that contains a `log.txt` file).\n- Because the application later reads those constructed paths with `fopen`, an attacker can cause the application to open and disclose files outside of the `./data` directory — limited to files with the final filename `log.txt` or `secret.txt` (depending on which endpoint is used).\n- Combined with other weaknesses (weak permissions on `./data`, or other application logic), this increases risk of information disclosure or tampering.\n\n## Impact\n\n- Information disclosure: reading files outside the application's intended directory tree that are named `log.txt` or `secret.txt` (or placing an attacker-controlled directory containing `log.txt` accessible via traversal) may disclose sensitive data.\n- Local filesystem manipulation (if the attacker can also cause writes) can lead to tampering.\n\n## Steps to Reproduce / Exploit\n\n1. Connect to the service on the TCP endpoint (e.g., `nc <host> 1781`).\n2. Choose `2` (View a previous log) or `3` (View an unredacted log).\n3. When prompted for `Enter log ID:`, provide a traversal string such as `../../some/dir`.\n4. The application will attempt to open `./data/../../some/dir/log.txt` which resolves to `../some/dir/log.txt` in the filesystem; if that file exists and is readable by the process, the contents will be shown.\n\nNotes: Since the code unconditionally appends `log.txt` or `secret.txt`, the traversal allows access to files that end with those names under other directories. An attacker that can place files on the host (or guess existing directories that contain `log.txt`) can leverage this.\n\n## Remediation\n\n- Sanitize and validate IDs: only accept a strict character set (e.g., [A-Za-z0-9_-]) and enforce a reasonable maximum length.\n- Reject or canonicalize inputs that contain `/` or `..` path components before using them in path construction.\n- Use secure directory APIs that check that the final resolved path is inside the intended data directory (e.g., resolve to an absolute path and verify it has the expected prefix) before opening files.\n- When possible, store logs with an internal mapping (database or index) rather than treating the user-supplied token as a direct filesystem path component.\n",
#           "summary": "User-supplied log IDs are concatenated into filesystem paths without sanitization; `..` and slashes allow path traversal to other filesystem locations and potential file disclosure.",
#           "file_path": "/opt/resources/logger",
#           "confidence": "high",
#           "severity": "medium",
#           "exploitability": "medium"
#         },
#         {
#           "title": "Insecure temporary file handling and overly permissive directory permissions (TOCTOU / symlink risk)",
#           "report": "## Overview\n\nThe application creates per-log directories with mode 0777 and performs file replacement using predictable temporary filenames (`<path>.tmp`) followed by `rename()`. This pattern, combined with the predictable temporary filename and world-writable directories, can be abused by local attackers to perform TOCTOU (time-of-check/time-of-use) or symlink attacks to overwrite or read arbitrary files.\n\n## Where it occurs\n\n- Binary: `/opt/resources/logger` (decompiled source)\n- Locations:\n  - Directory creation in `save_log()` using `mkdir(local_1158, 0x1ff)` (0x1ff = 0777)\n  - Temporary file creation in `redact_log()`:\n\nRelevant snippets:\n\n```c\n/* save_log */\nsnprintf(local_1158,0x20, \"%s%s\", \"./data/\", local_1198);\nmkdir(local_1158, 0x1ff); // 0777\n...\n/* redact_log */\nsnprintf(local_1058, 0x40, \"%s.tmp\", param_1);\n__stream_01 = fopen(local_1058, \"w\");\n...\nrename(local_1058, param_1);\n```\n\n## Vulnerability Details\n\n- The directory for each log is created with permissions `0777` (read/write/execute for owner, group, others). That makes it possible for other local users or processes to create files or symlinks inside that directory if they have access to the parent `./data` directory.\n- The temporary filename used for redaction is deterministic (`<path>.tmp`). Opening and writing the temp file with `fopen(..., \"w\")` without creating it securely (no exclusive flag or atomic temporary creation) allows an attacker who can create a symlink in place of the temp filename to cause the program to write to an arbitrary location.\n- After writing the temp file, the program calls `rename(temp, final)` to replace the original file. A symlink race or TOCTOU can cause `rename()` to affect a different target than intended, potentially overwriting arbitrary files that the process has permissions to write.\n\n## Impact\n\n- Local privilege escalation or tampering: on systems where multiple unprivileged users share the same host or the `./data` directory is accessible to other parties, an attacker could cause the service to overwrite files or write attacker-controlled content to sensitive locations that the service can write.\n- Data integrity and confidentiality loss for log files and related artifacts.\n\n## Steps to Reproduce / Exploit\n\n1. On the host where `./data` is mounted, create a directory or symlink with a predictable target name that the service will use (or exploit the fact that directories are created world-writable to create the temp filename before the service).\n2. Trigger `save_log()` (remote) which will call `redact_log()` and attempt to create `<path>.tmp`. If a symlink or file already exists at that path pointing elsewhere, the service will open and write to the attacker-controlled target.\n3. After `rename()` executes, the attacker-controlled file may be moved into place as the service's final log file, allowing overwrite of a file the service controls.\n\nThis is primarily a local attack but is exploitable in multi-tenant host environments or when host directories are shared.\n\n## Remediation\n\n- Create directories with the minimum required permissions (e.g., `0700` or `0750`) rather than `0777`.\n- Use secure temporary file creation APIs (e.g., `mkstemp`) that create temporary files atomically and with exclusive access.\n- Use `open(..., O_CREAT | O_EXCL, mode)` when creating files and avoid predictable temporary filenames.\n- Consider performing file operations as an unprivileged user with a private data directory that is not world-writable or shared.\n",
#           "summary": "The service creates per-log directories with 0777 permissions and uses predictable .tmp filenames and rename(), allowing TOCTOU/symlink attacks by local users or co-tenants.",
#           "file_path": "/opt/resources/logger",
#           "confidence": "high",
#           "severity": "medium",
#           "exploitability": "medium"
#         },
#         {
#           "title": "Predictable ID/secret generation using srand()/rand()",
#           "report": "## Overview\n\nThe service generates public IDs and per-log secrets using the C library PRNG (`srand()`/`rand()`) seeded with a combination of `time()`, `getpid()`, and a global counter. This approach is predictable and not suitable for security-sensitive identifiers or secrets; an attacker who can approximate the process start time and PID may be able to guess or brute-force generated values.\n\n## Where it occurs\n\n- Binary: `/opt/resources/logger` (decompiled source)\n- Function: `generate_id(char *param_1, size_t param_2)`\n\nDecompiled excerpt:\n\n```c\nvoid generate_id(char *param_1,size_t param_2)\n{\n  time_t tVar3 = time(NULL);\n  uint uVar1 = getpid();\n  uVar1 = counter ^ (uint)tVar3 ^ uVar1;\n  counter = counter + 1;\n  srand(uVar1);\n  int r = rand();\n  snprintf(param_1, param_2, \"%012ld\", (long)r % 1000000000000);\n}\n```\n\n## Vulnerability Details\n\n- `srand()` seeded with (time ^ pid ^ counter) is predictable to an attacker who can approximate the process time and PID (both low-entropy values).\n- `rand()` is not cryptographically secure and should not be used for generating secrets or identifiers that must be unguessable.\n- While `save_log()` prints the generated secret to the client who created the log (so that client can later un-redact), if there were any scenario relying on `generate_id()` for secrecy (e.g., short-lived tokens), those values could be predicted.\n\n## Impact\n\n- If any security relies on unpredictability of the generated IDs/secrets, this predictability reduces their effective entropy and enables brute force or guess attacks.\n- An attacker with additional knowledge (exact server time, PID pattern) can reduce the search space and guess generated values faster.\n\n## Steps to Reproduce / Exploit\n\n1. Observe the application behavior or create one or more logs to gather timestamps and patterns of generated IDs.\n2. Approximate or enumerate the `time()` value and potential PIDs and counter states used when the server generated identifiers.\n3. Re-seed a local PRNG with candidate seeds and generate `rand()` outputs; compare to observed IDs to find matches.\n\nThis allows narrowing the search and enumerating potential IDs/secrets with fewer attempts than brute force.\n\n## Remediation\n\n- Use a cryptographically secure random number generator for any identifiers or secrets. On Linux, use `/dev/urandom` or functions such as `getrandom()` or a secure library (e.g., libsodium or OpenSSL RAND_bytes).\n\nExample replacement:\n\n```c\n#include <unistd.h>\n#include <fcntl.h>\n\nint fd = open(\"/dev/urandom\", O_RDONLY);\nunsigned char buf[8];\nread(fd, buf, sizeof(buf));\nuint64_t v = *(uint64_t*)buf;\nclose(fd);\nsnprintf(param_1, param_2, \"%012lu\", v % 1000000000000UL);\n```\n\n- If display of the secret to the user is necessary, do so, but ensure the secret is generated using cryptographically secure RNG and that access controls prevent unauthorized discovery.\n",
#           "summary": "IDs/secrets are generated using srand()/rand() with a time/pid-based seed — predictable and unsuitable for secret generation.",
#           "file_path": "/opt/resources/logger",
#           "confidence": "high",
#           "severity": "low",
#           "exploitability": "low"
#         }
#       ] + [
#         {
#           "title": "Remote stack buffer overflow via unchecked scanf(\"%s\") in logger",
#           "report": "# Overview\n\nThe logger binary exposes an interactive menu over a TCP connection (socat -> /service/logger). Multiple menu options accept user-controlled strings using scanf(\"%s\", ...) into fixed-size stack buffers without any width limit. This results in a classic stack-based buffer overflow that can be triggered remotely via the network service, allowing memory corruption and potentially remote code execution (RCE).\n\n# Where it occurs\n\nThe issue is present in the decompiled logger functions that handle user input for log IDs and secrets. Relevant locations in the decompiled output (binary: /opt/resources/logger) include the view_log() and view_unredacted() input handling code.\n\nExample decompiled snippets (from /opt/resources/logger / decompiled to /tmp/decompile_output/out):\n\n```c\n/* view_log */\nchar local_1058[32];\nchar local_1038[32];\n...\nprintf(\"Enter log ID: \");\nfflush(stdout);\nscanf(\"%s\", local_1058);   // <-- unsafe: no width restriction\ngetchar();\nsnprintf(local_1038,0x20,\"%s%s/log.txt\",\"./data/\",local_1058);\n__stream = fopen(local_1038,\"r\");\n```\n\n```c\n/* view_unredacted */\nchar local_20a8[32];\nchar local_2088[32];\n...\nprintf(\"Enter log ID: \");\nfflush(stdout);\nscanf(\"%s\", local_20a8);    // <-- unsafe: no width restriction\ngetchar();\nprintf(\"Enter redacted secret: \");\nfflush(stdout);\nscanf(\"%s\", local_2088);    // <-- unsafe: no width restriction\ngetchar();\n```\n\n# Vulnerability Details\n\nscanf(\"%s\", buf) reads a whitespace-delimited string from input and places it into the provided buffer, but it does not enforce a maximum length. If an attacker provides more bytes than the buffer was allocated for (32 bytes in the examples), scanf will write beyond the buffer bounds, clobbering adjacent stack data (saved frame pointer, saved return address, stack canary area, other locals).\n\nThe decompiled binary shows the use of stack-protector semantics (calls to __stack_chk_fail in functions), so a classic return-address overwrite may require bypassing or leaking the stack canary. Still, buffer overflow allows memory corruption and is a severe vulnerability: it can be combined with information disclosure or use-after-free/heap vulnerabilities to gain code execution, or abused to crash the service (denial of service).\n\nBecause the program is executed by socat on a TCP listener (port 1781 in Dockerfile), this vulnerability is reachable remotely by any client that connects to the service and interacts with the menu.\n\n# Impact\n\n- Remote Denial of Service: crafted input will crash the process.\n- Potential Remote Code Execution (RCE): with additional technique (canary leak bypass, ROP), an attacker could execute arbitrary code as the user running the service (container root in this image), gaining full control over the container filesystem and any mounted volumes.\n- Data exposure and lateral movement: attacker could read or overwrite files under ./data/, exfiltrate secrets, or use the container to pivot further.\n\n# Steps to Reproduce / Exploit\n\n1. Start the service (as in Dockerfile: socat TCP-LISTEN:1781,fork,reuseaddr EXEC:\"/service/logger\").\n2. Connect with netcat or telnet: nc <host> 1781\n3. The menu will be printed; choose the option that triggers the vulnerable scanf (option 2 for view_log or option 3 for view_unredacted). Example sequence (simplified):\n\n```\n2\\n                # select \"View a previous log\"\n<very long string >\\n  # supply >32 bytes as the \"log ID\" input\n```\n\n4. Observe process crash or unexpected behavior when the input length exceeds the expected 32-byte buffer.\n\nTo escalate to RCE, an attacker would need to:\n- Leak the stack canary (e.g., via another bug or information disclosure) or find a path that does not check the canary.\n- Build a ROP chain or ret2libc payload and overwrite the saved return address accordingly.\n\nA minimal PoC that demonstrates crash is to send a menu selection and then an input of 100+ 'A' characters in place of the log ID; the binary should terminate or exhibit abnormal behavior.\n\n# Remediation\n\n- Do not use scanf(\"%s\", buf). Use bounded reads that accept a maximum length. For example, change scanf(\"%s\", buf) to fgets(buf, sizeof buf, stdin) (and then strip the trailing newline), or use scanf(\"%31s\", buf) to limit input to the buffer size minus one.\n\nExample fix:\n\n```c\nchar buf[32];\nif (fgets(buf, sizeof(buf), stdin) != NULL) {\n    size_t n = strcspn(buf, \"\\n\");\n    buf[n] = '\\0';\n}\n```\n\n- Validate input length before use.\n- Consider using safer APIs or higher-level parsing libraries.\n- Consider dropping privileges in the container (do not run as root) and use filesystem permissions to limit potential impact.\n- Employ AddressSanitizer and fuzzing in CI to catch similar issues early.\n",
#           "summary": "Remote stack buffer overflow in logger via unchecked scanf(\"%s\") in view_log/view_unredacted, reachable over TCP. Can lead to DoS and possibly RCE.",
#           "file_path": "logger",
#           "confidence": "certain",
#           "severity": "high",
#           "exploitability": "high"
#         },
#         {
#           "title": "Sensitive data exposure: secret ID written to and displayed in public log",
#           "report": "# Overview\n\nThe logger service writes a generated \"secret\" identifier into the public log and prints it to the user when the log is saved. The same public log can later be viewed by anyone who knows the public ID (via the unauthenticated \"View a previous log\" menu option), which leaks the secret. Additionally, the logic for viewing unredacted logs incorrectly uses the redaction regex (saved in secret.txt) as the authentication token instead of the generated secret. This results in inconsistent and insecure handling of secrets and puts sensitive data at risk.\n\n# Where it occurs\n\nThis occurs primarily in save_log(), redact_log(), and view_log()/view_unredacted() in the logger binary (decompiled output: /tmp/decompile_output/out). The relevant snippets are:\n\n```c\n/* save_log */\ngenerate_id(local_1198,0x20);   // public ID\ngenerate_id(local_1178,0x20);   // secret ID\n...\n__stream = fopen(\"./data/<id>/log.txt\",\"w\");\nfprintf(__stream, \"%s\\n\", local_1178);  // secret written to log.txt\n...\nprintf(\"Log saved with public ID %s and redacted secret %s\\n\", local_1198, local_1178);\n```\n\n```c\n/* redact_log */\n__stream_00 = fopen(\"./data/<id>/secret.txt\",\"w\");\nregcomp(&local_1098,param_3,1);\nfprintf(__stream_00, \"%s\\n\", param_3);  // writes the regex pattern into secret.txt\n// later writes found matches into secret.txt as well\n```\n\n```c\n/* view_log */\nscanf(\"%s\", local_1058);\nsnprintf(local_1038,0x20,\"%s%s/log.txt\",\"./data/\",local_1058);\n__stream = fopen(local_1038,\"r\");\n// prints full log contents (including first line which contains the secret ID)\n```\n\n```c\n/* view_unredacted */\nscanf(\"%s\", local_20a8);          // log id\nscanf(\"%s\", local_2088);          // redacted secret (user-supplied)\n...\nfgets(local_2028,0x1000,__stream); // read first line of secret.txt\nsVar2 = strcspn(local_2028, \"\\n\"); local_2028[sVar2] = '\\0';\niVar1 = strcmp(local_2028, local_2088);  // compares user input to secret.txt first line\n```\n\n# Vulnerability Details\n\n- The service stores the generated \"secret ID\" (local_1178) directly in the public log (log.txt first line) and prints the secret to the user's console when creating a log. The public viewer (view_log) will read and display that file to any client who supplies the public ID.\n\n- The expected authorization model appears to be that the generated secret (local_1178) should control access to unredacted logs. However, view_unredacted compares the first line of secret.txt (which is the regex pattern, saved by redact_log) to the user-supplied \"redacted secret\" input. Thus the code verifies the regex pattern rather than the intended secret identifier, producing an authentication logic bug.\n\n- This is an information exposure problem and an authentication/authorization flaw: secrets are stored and exposed in plaintext and the auth check is inconsistent with how secrets are issued.\n\n# Impact\n\n- Anyone who can connect to the service can create logs and will receive and see the secret. More importantly, anyone who knows or can enumerate a public ID can view the public log (via view_log) and read the secret ID because the secret was written as the first line of the log.\n\n- Attackers can leverage this to learn sensitive secret IDs and then attempt to misuse them (or combine this with other flaws) to access additional content.\n\n- The mismatched authentication check (pattern vs secret) prevents the secret from acting as the intended auth token, meaning the real protection of unredacted data is broken by design. Meanwhile, redact_log writes sensitive matched substrings to secret.txt in plaintext (see another finding), compounding the exposure.\n\n# Steps to Reproduce / Exploit\n\n1. Connect to the service on TCP port 1781 (socat is configured in the Dockerfile to run the binary on that port).\n2. Choose the menu option to \"Save a new log\" and follow prompts to enter log contents and a redaction regex.\n3. The service responds with a message such as: \n\n```\nLog saved with public ID 000123456789 and redacted secret 987654321000\n```\n\n4. Any client can now do option 2 (View a previous log) and provide the public ID (000123456789) to receive the full log contents. The first line of the log contains the \"redacted secret\" value (987654321000), which is displayed and therefore leaked to any requester.\n\n5. Because the program uses the regex as the first line of secret.txt (not the secret ID), an attacker who wishes to access unredacted logs cannot simply provide the printed secret to the view_unredacted prompt—the logic is inconsistent.\n\n# Remediation\n\n- Do not write authentication secrets into publicly readable artifacts (logs). The first line of log.txt should not contain any secret tokens.\n\n- Do not print secrets back to the user in an unauthenticated or public context. If a secret must be displayed once, ensure it is only shown to the creator and not written to persistent files.\n\n- Fix the authentication logic: choose a single secret that controls access to unredacted logs (e.g., local_1178), store it securely (e.g., hashed if necessary), and verify that token in view_unredacted. Do not use the redaction pattern as an auth token.\n\n- Use least-privilege file permissions when creating directories and files. Remove world-writable creation modes (see separate finding on file permissions).\n\n- Consider an explicit access control mechanism rather than relying on \"security through obscurity\" by IDs.\n",
#           "summary": "The service writes the secret token into the public log and prints it when creating logs; anyone viewing the public log can read the secret. The auth check for unredaction incorrectly uses the redaction regex rather than the intended secret token.",
#           "file_path": "logger",
#           "confidence": "high",
#           "severity": "high",
#           "exploitability": "high"
#         },
#         {
#           "title": "Unsafe redaction: secret matches written to secret.txt and regex may cause ReDoS",
#           "report": "# Overview\n\nThe redaction implementation is fundamentally flawed: rather than removing sensitive substrings and discarding them, the service writes the user-supplied redaction pattern and every matched substring (the sensitive data) to a secret file (./data/<id>/secret.txt) in plaintext. Additionally, the service accepts an arbitrary regular expression from the user and compiles/runs it with no limits, allowing a crafted pattern to trigger catastrophic backtracking and CPU exhaustion (a ReDoS - Regular Expression Denial of Service).\n\n# Where it occurs\n\nIn redact_log() in the logger binary (decompiled: /tmp/decompile_output/out). Relevant code snippets:\n\n```c\nregcomp(&local_1098, param_3, 1);\nfprintf(__stream_00, \"%s\\n\", param_3);  // writes regex pattern into secret.txt\nwhile (fgets(local_1018,0x1000,__stream) != NULL) {\n    while (regexec(&local_1098, local_1018, 1, &local_10a8, 0) == 0) {\n        fprintf(__stream_00, \"%.*s\\n\", (ulong)(uint)((int)local_10a0 - local_10a8.rm_so), \n                local_1018 + (long)local_10a8);\n        memset(local_1018 + (long)local_10a8, 0x2a, local_10a0 - (long)local_10a8); // replace match with '*'\n    }\n    fputs(local_1018, __stream_01);\n}\n```\n\n# Vulnerability Details\n\n1) Sensitive data stored in plaintext: The code intentionally writes each found match to secret.txt. That file therefore contains the very data the redaction step is supposed to protect. If an attacker or another process can read secret.txt (or if the application later reveals it via a bug), the sensitive data are exposed.\n\n2) Flawed authentication model: secret.txt’s first line is the regex pattern (param_3); view_unredacted compares the user-supplied token to this first line (pattern), not to a stored secret token. This compound logic error creates confusion and weak protection of sensitive data.\n\n3) ReDoS (RegExp DoS): regcomp/regexec are invoked with a user-supplied pattern and no limits on time or input size. A maliciously crafted pattern (e.g., nested quantifiers) or a pathological log line can cause regexec to perform heavy backtracking and consume CPU for an extended time while processing a single connection. Since the binary is invoked per connection by socat and runs with the privileges of the container, a single client can tie up the container CPU and cause denial-of-service for other clients.\n\n# Impact\n\n- Plaintext exfiltration of sensitive substrings: secret.txt contains both the regex and the matched substrings; if that file can be read (by local attackers, via file-symlink attacks, or via future code bugs), sensitive data will be leaked.\n\n- Denial of Service: remote clients can provide expensive regexes or patterns that cause catastrophic backtracking against large log lines, exhausting CPU and making the service unresponsive.\n\n# Steps to Reproduce / Exploit\n\n1) Create a log using the service and supply a pattern that will match sensitive content; inspect the created files under ./data/<id>/secret.txt (or use the service to trigger code paths that reveal secret.txt). You will find the pattern on the first line and all matched substrings on subsequent lines.\n\n2) For ReDoS: provide a pathological regex (e.g., (a+)+$ or similar) and then submit a long input line that triggers catastrophic backtracking. Observe that the process will hang or consume heavy CPU on regexec calls while processing the line.\n\nNote: Because the service spawns one /service/logger process per connection, a single malicious client can use this to consume CPU.\n\n# Remediation\n\n- Do not write matched sensitive substrings to storage in plaintext. If you must keep a record of redacted data, store only secure cryptographic hashes or encrypt the data using a strong algorithm and protect keys with proper access control.\n\n- Fix the authentication model: the authentication token (secret ID) should be generated and stored securely, separate from the redaction pattern. Verify that the correct token is required for unredaction.\n\n- Consider a safer redaction approach: instead of using arbitrary user regexes, build a controlled set of built-in patterns or sanitize/limit regex complexity. Alternatively, set timeouts or evaluation limits on regex execution (e.g., use a regex engine that supports iterative matching with bounded backtracking or that exposes a run-time limit).\n\n- Validate and limit the size of input lines and the compiled regex (e.g., maximum pattern length) to reduce attack surface.\n\n- Use secure file permissions (do not create world-writable files) and consider ephemeral storage or in-memory redaction to avoid persisting sensitive plaintext.\n",
#           "summary": "Redaction implementation stores matched sensitive substrings and the regex plaintext in secret.txt and accepts unbounded user regexes (ReDoS risk).",
#           "file_path": "logger",
#           "confidence": "high",
#           "severity": "high",
#           "exploitability": "high"
#         },
#         {
#           "title": "Predictable ID generation: use of srand()/rand() for public/secret IDs",
#           "report": "# Overview\n\nThe logger uses a custom generate_id() function that seeds the C PRNG (srand) using a value derived from current time, PID, and an internal counter, and then uses rand() to produce a numeric ID string. This approach is not cryptographically secure and allows an attacker with modest knowledge (e.g., approximate creation time, process characteristics) to predict or brute-force generated IDs for public or secret tokens.\n\n# Where it occurs\n\nIn the decompiled function generate_id() in the logger binary (/opt/resources/logger decompiled in /tmp/decompile_output/out):\n\n```c\nvoid generate_id(char *param_1,size_t param_2)\n{\n  uint uVar1;\n  int iVar2;\n  time_t tVar3;\n\n  tVar3 = time((time_t *)0x0);\n  uVar1 = getpid();\n  uVar1 = counter ^ (uint)tVar3 ^ uVar1;\n  counter = counter + 1;\n  srand(uVar1);\n  iVar2 = rand();\n  snprintf(param_1,param_2,\"%012ld\",(long)iVar2 % 1000000000000);\n  return;\n}\n```\n\n# Vulnerability Details\n\n- srand()/rand() provide a pseudo-random number generator that is not suitable for generating secrets or identifiers that must remain unpredictable. The seed (uVar1) is computed from the current time (t), process id (pid) and an incremental counter; these values are easily approximated or partially known by an attacker. reseeding srand before each call is also bad practice: it reduces the entropy and makes the output highly predictable.\n\n- The generated ID is then printed to the client and/or stored. The code uses generate_id() twice: once for a public ID and once for a \"secret\" ID. Using the same predictable approach for the secret token undermines its security guarantees.\n\n# Impact\n\n- Attackers can predict or brute-force secret IDs, enabling enumeration of logs or unauthorized access to log data.\n\n- Where IDs are relied on for authentication or access control, the weak generation makes the application vulnerable to guessing attacks and account/file enumeration.\n\n# Steps to Reproduce / Exploit\n\n1. Observe how IDs are generated by creating a few logs and collecting the public IDs and their associated timestamps.\n2. Using the knowledge of the approximate time and typical PID space (or by creating logs yourself), reconstruct or brute-force the PRNG outputs and predict future/secrets IDs.\n\nEven without perfect reconstruction, the small keyspace (12-digit decimal mod value) and predictable seeding greatly reduce the search difficulty.\n\n# Remediation\n\n- Use a cryptographically secure source of randomness for any identifiers or secrets. For example, use getrandom()/getentropy()/arc4random_buf() or read from /dev/urandom, and format the output or use a secure token library.\n\nExample using getrandom()//dev/urandom (POSIX):\n\n```c\nunsigned char buf[16];\nif (getrandom(buf, sizeof(buf), 0) == sizeof(buf)) {\n    // convert to hex or base64 string\n}\n```\n\n- Do not reseed the PRNG on every call. If using a PRNG that must be seeded, seed it once with a high-entropy seed at process start and use a secure RNG for secrets.\n\n- Increase token length and entropy (use >=128-bit randomness for secrets). Avoid using decimal-only small ranges as secret tokens.\n\n- Consider rotating secrets and invalidating old tokens where feasible.\n",
#           "summary": "IDs and secret tokens are generated using srand()/rand() seeded with time/pid/counter, making them predictable. Use a cryptographically secure RNG.",
#           "file_path": "logger",
#           "confidence": "high",
#           "severity": "medium",
#           "exploitability": "medium"
#         },
#         {
#           "title": "Insecure file handling: world-writable directories and unsafe rename/fopen (symlink/TOCTOU risk)",
#           "report": "# Overview\n\nThe service creates directories and files under ./data/ with permissive mode bits and performs non-atomic file operations (fopen + rename) without validating or protecting against symlink attacks or race conditions. This allows local or malicious users with access to the container (or other processes in the same environment) to perform symlink or TOCTOU attacks to read or overwrite sensitive files.\n\n# Where it occurs\n\nIn save_log() and redact_log() in the decompiled logger binary (/opt/resources/logger). Example lines:\n\n```c\nsnprintf(local_1158,0x20,\"%s%s\",\"./data/\",local_1198);\nmkdir(local_1158, 0x1ff);   // mode 0777\n...\nsnprintf(local_1058,0x40, \"%s.tmp\", param_1);\n__stream_01 = fopen(local_1058, \"w\"); // create temporary file\n...\nfclose(__stream_01);\nrename(local_1058, param_1); // replace original\n```\n\n# Vulnerability Details\n\n- mkdir(..., 0x1ff) creates the per-log directory with mode 0777 (world readable/writable/executable) which increases the risk that other untrusted local processes or users in the environment can create files or symlinks inside the directory.\n\n- The code uses fopen() to create temporary files and later uses rename() to atomically replace the original, but it does not use safe flags (open with O_EXCL/O_CREAT/ O_NOFOLLOW) or check ownership/permissions of existing files. If an attacker can race the creation of these files or place symlinks at the expected locations, fopen/rename may create or overwrite unintended targets.\n\n- If the container shares volumes or if other users/processes can write to the parent ./data/ directory, symlink attacks can be used to cause the application to write or move data into files outside the intended path (for instance redirecting the .tmp path to /etc/passwd or another sensitive file), leading to privilege escalation or file corruption.\n\n# Impact\n\n- Local file overwrite or arbitrary file write within the container, including potentially sensitive configuration or credential files.\n\n- Corruption of log data, or injection of attacker-controlled data into files the application subsequently trusts.\n\n- Combined with other vulnerabilities (e.g., remote code execution), an attacker can use the permissive file operations to persist or propagate further control.\n\n# Steps to Reproduce / Exploit\n\n1. If you have access to the container filesystem, create a directory or symlink at the expected path (./data/<id>/log.txt.tmp) pointing to a target file you want to overwrite.\n2. Trigger the service to create the log and perform the rename; the rename will follow the symlink, overwriting the target file.\n\nBecause the application sets the directory mode to 0777, other unprivileged processes in the container may be able to perform this attack if they can write under ./data/.\n\n# Remediation\n\n- Do not create directories with mode 0777. Use a more restrictive mode (e.g., 0755 or 0700) and consider using umask to restrict default file permissions.\n\n- When creating temporary files, use secure APIs and flags that avoid following symlinks (e.g., open with O_CREAT | O_EXCL | O_NOFOLLOW and check ownership). Use mkstemp(3) or openat with O_NOFOLLOW to safely create files in a race-resistant manner.\n\n- Validate the path before writing and avoid assuming user-supplied IDs map to safe filesystem paths. Consider canonicalizing or verifying that the resulting path remains under the intended base directory before performing file operations.\n\n- Run the service with least privilege (do not run as root inside the container) and isolate storage volumes to prevent other local users from being able to tamper with ./data/.\n",
#           "summary": "Service creates directories with 0777 and uses fopen/rename without safeguards, enabling symlink/TOCTOU attacks and arbitrary file overwrite within the container.",
#           "file_path": "logger",
#           "confidence": "high",
#           "severity": "medium",
#           "exploitability": "medium"
#         },
#         {
#           "title": "Insecure container configuration: service runs as root and exposes logger via socat",
#           "report": "# Overview\n\nThe Dockerfile runs the logger directly under the root user and uses socat to expose the binary on a TCP port. Running network-facing binaries as root increases the impact of any vulnerability in the binary—successful exploitation will yield root access to the container. Additionally, the use of socat to spawn the binary per-connection expands the attack surface.\n\n# Where it occurs\n\nDockerfile in the repository (/opt/resources/Dockerfile) contains:\n\n```dockerfile\nFROM alpine:3.19\n...\nWORKDIR /service\nCOPY logger /service/logger\nRUN chmod +x /service/logger\nEXPOSE 1781\nCMD socat TCP-LISTEN:1781,fork,reuseaddr EXEC:\"/service/logger\",stderr\n```\n\nNo USER directive is present; the container executes as root by default. socat is used to accept incoming TCP connections and execute /service/logger for each connection.\n\n# Vulnerability Details\n\n- Running the service as root increases privileges available to an attacker who compromises the process. Combined with the host-level misconfigurations (e.g., privileged mounts) or other container escape issues, this can substantially increase the risk.\n\n- socat spawns a fresh /service/logger process per connection; this is convenient but ensures every vulnerability in the binary is directly reachable from the network. If the binary has memory corruption bugs (as found elsewhere), the consequence is immediate root-level command execution within the container.\n\n# Impact\n\n- Compromise of the container as root on successful exploitation of any RCE/escape vulnerability in the logger.\n\n- Increased blast radius for any stored secrets or host-bound mounts.\n\n# Steps to Reproduce / Exploit\n\n1. Build and run the Docker container as provided. Observe that it runs as root.\n2. Exploit a vulnerability in the network-facing binary (e.g., buffer overflow described earlier) to execute commands — those commands will run as root in the container.\n\n# Remediation\n\n- Run the service as a non-root user inside the container. Add a non-privileged user in the Dockerfile and switch to it with USER.\n\nExample:\n\n```dockerfile\nRUN addgroup -S appgroup && adduser -S appuser -G appgroup\nCOPY logger /service/logger\nRUN chown -R appuser:appgroup /service\nUSER appuser\nCMD socat TCP-LISTEN:1781,fork,reuseaddr EXEC:\"/service/logger\",stderr\n```\n\n- Alternatively, avoid using socat and instead run a hardened network server that drops privileges and enforces request limits.\n\n- Reduce container capabilities and use runtime confinement (seccomp, AppArmor, SELinux) to minimize impact of compromise.\n",
#           "summary": "Container runs as root (no USER set) and exposes the logger via socat. Successful exploitation of network bugs yields root in the container.",
#           "file_path": "Dockerfile",
#           "confidence": "high",
#           "severity": "medium",
#           "exploitability": "medium"
#         }
#       ]

# resp = requests.post(f'{url}/jobs', json={
#     'org': "logger",
#     'repo': "logger",
#     'job_type': "combiner-analysis",
#     "payload": {
#         'enqueue_exploit_analysis': True,
#         'findings': findings,
#         'add_findings': True,
#     }
# })
# print(resp.status_code, resp.text)

# payload = {
#     "owner": "logger",
#     "repo": "logger",
#     "issue_number": 47,
#     "finding": {
#       "title": "Insecure file handling and path-traversal: predictable .tmp, mkdir 0777, and unsanitized log IDs",
#       "report": "## Overview\n\nThe `logger` binary builds filesystem paths directly from user-supplied log IDs and uses predictable temporary filenames and permissive directory permissions when manipulating files. These issues combine to allow path traversal reads, symlink/TOCTOU races and arbitrary file overwrite or disclosure when an attacker can place files or symlinks in the `./data` tree (which is host-mounted by default). The service constructs paths using `snprintf(\"./data/%s/log.txt\", id)` with no sanitization and creates temporary files using a deterministic `<path>.tmp` name before `rename()`-ing them into place.\n\n## Where it occurs\n\n- Binary: `/opt/resources/logger` (decompiled)\n- Functions / locations: `save_log()`, `redact_log()`, `view_log()`, `view_unredacted()`\n\nDecompiled excerpts:\n\n```c\n/* path construction without filtering */\nsnprintf(local_1158,0x20,\"%s%s/log.txt\",\"./data/\", local_1198);\n__stream = fopen(local_1158, \"w\");\n\n/* world-writable directory creation */n\nmkdir(local_1158, 0x1ff); /* 0x1ff == 0777 */\n\n/* predictable temporary file used for redaction replacement */\nsnprintf(local_1058,0x40, \"%s.tmp\", param_1);\n__stream_01 = fopen(local_1058, \"w\");\n... \nrename(local_1058, param_1);\n```\n\nAlso, `view_log()` and `view_unredacted()` read the log ID directly from user input using `scanf(\"%s\", local_1058);` and then construct `./data/<id>/log.txt` and `./data/<id>/secret.txt` with no validation.\n\n## Vulnerability Details\n\n1) Path traversal: Because the log ID is used verbatim as a filename component, an attacker can include `..` or path separators (`/`) to cause the code to open files outside the intended `./data` folder (e.g., `./data/../../etc/secretdir/log.txt`). The code appends `log.txt` or `secret.txt`, so an attacker can target files that end with those names under other directories.\n\n2) TOCTOU / symlink races: The redaction code writes to a temporary filename derived deterministically from the target path (`<path>.tmp`) and uses `fopen(..., \"w\")` and later `rename()` to replace the original file. If an attacker can create a symlink at the predictable `.tmp` name (or can race between file creation and open), the service may write into an attacker-controlled target or overwrite arbitrary files.\n\n3) World-writable directories: `mkdir(..., 0x1ff)` sets `0777` permissions on per-log directories, enabling other local users or processes to create files or symlinks inside `./data` and facilitating the attacks above.\n\n## Impact\n\n- Information disclosure: path traversal could cause the application to read files outside `./data` that match the `log.txt`/`secret.txt` naming scheme.\n- Arbitrary file overwrite: symlink/TOCTOU attacks may allow an attacker to make the service write into arbitrary files that the service user can write to, potentially causing data corruption or privilege escalation in multi-tenant environments.\n- Persistence misuse: because `./data` is host-mounted by default, host users may be able to create entries that the service will process incorrectly.\n\n## Steps to Reproduce / Exploit\n\n1) Path traversal read: Connect to the service, choose `View a previous log` and supply an ID such as `../../some/dir` — the program will attempt to open `./data/../../some/dir/log.txt`, which resolves to `../some/dir/log.txt` on the host. If that file exists and is readable, its content will be shown.\n\n2) Symlink/TOCTOU overwrite: On the host (or if you can write into `./data`), create a symlink named `./data/<id>/log.txt.tmp` pointing at a target you want to overwrite. Trigger `save_log()`/`redact_log()` remotely — the service will `fopen()` the predictable `.tmp` path and write attacker-controlled content to the symlink target, then call `rename()`.\n\n## Remediation\n\n- Sanitize and validate log IDs: restrict allowable characters (e.g., [A-Za-z0-9_-]), enforce a strict maximum length, and reject any ID containing `/` or `..`.\n- Resolve and verify final paths: after constructing the path, resolve (realpath) and verify it is within the intended `./data` base directory before opening.\n- Use secure temporary file creation APIs: prefer `mkstemp()` or open with `O_CREAT|O_EXCL|O_NOFOLLOW` to avoid following symlinks and to create files atomically.\n- Tighten permissions: do not create directories with `0777`; use `0700` or `0750` and set a proper umask. Run the service as an unprivileged user so that other local users cannot abuse the data directory.\n- Avoid host mounting sensitive data by default. If a volume must be mounted, ensure host permissions prevent untrusted users from writing into it.\n\nApplying these mitigations reduces the risk of arbitrary file reads/writes, TOCTOU/symlink attacks and local escalation via filesystem races.\n",
#       "summary": "User-supplied IDs are used directly in filesystem paths; the app also creates world-writable directories and uses predictable .tmp filenames before rename(), enabling path traversal and TOCTOU/symlink attacks.",
#       "file_path": "logger",
#       "confidence": "high",
#       "severity": "medium",
#       "exploitability": "medium"
#     }
#   }

# resp = requests.post(f'{url}/jobs', json={
#     'org': "logger",
#     'repo': "logger",
#     'job_type': "exploit-analysis",
#     "payload": payload
# })
# print(resp.status_code, resp.text)

200 {"job_id":"421e448c-22b0-4487-8b2a-10b46a1bf1fe","status":"queued"}


In [ ]:
finding = {
    "title": "Predictable ID/secret generation via insecure srand()/rand() seeding",
    "report": "## Overview\n\nThe binary generates both the public log ID (directory name) and the per-log secret using the C standard library PRNG (srand()/rand()) and seeds it with predictable values (current time, process id and an internal counter). This produces non-cryptographically-random identifiers and secrets that can be predicted or brute-forced by an attacker. Secrets used to authorize unredaction are therefore not securely generated.\n\n## Where it occurs\n\nFunction `generate_id` in the `logger` binary (decompiled) seeds the PRNG and formats the output into a string buffer:\n\n```c\nvoid generate_id(char *param_1,size_t param_2)\n{\n  time_t tVar3 = time(NULL);\n  uint uVar1 = getpid();\n  uVar1 = counter ^ (uint)tVar3 ^ uVar1;\n  counter = counter + 1;\n  srand(uVar1);\n  int iVar2 = rand();\n  snprintf(param_1, param_2, \"%012ld\", (long)iVar2 % 1000000000000);\n}\n```\n\nThis function is called twice in `save_log()` to create a public directory ID and the corresponding secret value written to the log file first line and shown to the user.\n\n## Vulnerability Details\n\n- The implementation uses the non-cryptographic PRNG `rand()` seeded via `srand()`.\n- The seed is computed as a simple XOR of `time(NULL)`, `getpid()`, and an in-memory `counter` value. These values are either predictable (current time) or have a small entropy space (PID, counter).\n- Using this predictable seeding, an attacker who knows or can guess the approximate time a log was created (or can observe a related event) can re-seed an offline PRNG to reproduce the same sequence of rand() outputs and thus determine both the public ID and the secret.\n\nBecause the secret is used as the authentication token for viewing the unredacted log, a predicted secret allows bypassing the authentication check and reconstructing sensitive log contents.\n\n## Impact\n\n- An attacker that can guess or determine the approximate generation time (which is often trivial for temporally-correlated actions) can predict the secret and hence perform `view_unredacted` to recover the original unredacted log content.\n- If an attacker can predict or enumerate directory IDs, they may be able to access or enumerate stored logs.\n- Secrets created with this weak randomness are also vulnerable to offline brute-force where the entropy is low (rand() typically returns 31 bits or less).\n\nThis substantially weakens the intended secrecy of logs and allows unauthorized access to sensitive stored data.\n\n## Steps to Reproduce / Exploit\n\n1. Observe or estimate the time (wall-clock time) when logs are created. This could be done by creating your own logs and comparing times or by making repeated requests until the service responds.\n2. Reconstruct the seed computation in a local program using candidate times, known PIDs (musl often sets predictable PIDs in containers), and counter guesses (0,1,...).\n3. Seed a local PRNG using srand(seed) and call rand() to compute the expected outputs; format them with the same snprintf rules.\n4. If the predicted secret matches the secret stored in a target's log's first line, request `view_unredacted` supplying the guessed secret and the public ID to retrieve full unredacted contents.\n\nA simple brute-force script can iterate over a small time window and counter values to recover secrets quickly.\n\n## Remediation\n\n- Do not use `srand()`/`rand()` for generating secrets, IDs, or tokens. Use a cryptographically secure random source such as /dev/urandom, getrandom(), or platform CSPRNG APIs (e.g., arc4random, getrandom, OpenSSL RAND_bytes).\n\nExample (POSIX) using getrandom()/getentropy():\n\n```c\nunsigned char buf[16];\nif (getrandom(buf, sizeof(buf), 0) == sizeof(buf)) {\n    // convert to hex or base64 for ID/secret\n}\n```\n\n- Increase entropy (longer secrets) and do not expose raw secrets unnecessarily.\n- If secrets must be human-readable, use a proper token generation library (cryptographic GUIDs or HMAC-based tokens) and enforce sufficient length (128-bit+).\n- Log the use of random tokens only in secure locations and avoid printing them unnecessarily.\n\nFixing the RNG usage will prevent offline prediction and brute-force of authentication tokens and protect log confidentiality.\n",
    "summary": "generate_id() uses srand()/rand() seeded with time, PID, and a simple counter; this produces predictable IDs and secrets that can be guessed or brute-forced.",
    "file_path": "logger",
    "confidence": "high",
    "severity": "medium",
    "exploitability": "high"
}

payload = {
    "owner": "logger",
    "repo": "logger",
    "issue_number": 48,
    "finding": finding
}

resp = requests.post(f'{url}/jobs', json={
    'org': "logger",
    'repo': "logger",
    'job_type': "exploit-analysis",
    "payload": payload
})
print(resp.status_code, resp.text)

In [6]:
import re
def remove_suffix(s: str) -> str:
    return re.sub(r'-[0-9]+$', '', s)

remove_suffix("exampleasdasdasd212-3a")

'exampleasdasdasd212-3a'